In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'MIA': ['Norman Powell', 'Tyler Herro', 'Davion Mitchell'], 'WAS': ['Anthony Gill', 'Jaden Hardy', 'Tre Johnson', 'Justin Champagnie', 'Bilal Coulibaly'], 'BOS': ['Jaylen Brown'], 'IND': ['Jarace Walker', 'Kobe Brown', 'Ben Sheppard'], 'NYK': ['Tyler Kolek'], 'CHI': ['Matas Buzelis'], 'BKN': ['Noah Clowney', 'Terance Mann', 'Ziaire Williams', 'Josh Minott', 'Nolan Traore', 'Nic Claxton'], 'DAL': ['Marvin Bagley', 'Dwight Powell'], 'SAS': ['Stephon Castle', 'Victor Wembanyama'], 'DEN': ['Jamal Murray', 'Nikola Jokić', 'Christian Braun', 'Aaron Gordon', 'Cameron Johnson'], 'MIN': ['Anthony Edwards', 'Naz Reid'], 'MEM': ['Walter Clayton', 'Javon Small'], 'POR': ['Shaedon Sharpe', 'Vít Krejčí'], 'GSW': ['Al Horford', 'Kristaps Porziņģis', 'Will Richard', 'LJ Cryer', 'Gui Santos'], 'PHX': ['Jalen Green'], 'LAL': ['Jaxson Hayes', 'Marcus Smart']}

Out Players:
{'CLE': ['Thomas Bryant', 'Sam Merrill', 'Jarrett Allen', 'Donovan Mitchell'], 'ATL': ['Jock Landale'], 'MIA'

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
86,NaN,2025-26,1630573,Sam Hauser,Sam,1610612738,BOS,Boston Celtics,22501168,2026-04-09T00:00:00,BOS @ NYK,L,31.483333,2,7,0.286,2,6,0.333,0,0,0.000,0,2,2,3,0,0,0,0,0,0,6,1,12.9,0,0,13.0,1,31:29,1,118.6,119.0,119.0,114.9,115.3,115.3,3.8,3.7,3.7,0.143,0.00,30.0,0.000,0.071,0.032,0.0,0.0,0.429,0.429,0.101,0.104,89.46,89.19,74.33,89.19,0.047,58,2.0,7.0,38,84,0.452,16,43,0.372,14,16,0.875,13,29,42,23,11.0,4,1,2,16,17,106,-6.0,119.0,120.5,126.4,127.3,-7.4,-6.8,0.605,2.09,18.0,0.354,0.789,0.547,0.125,0.548,0.582,88.8,88.0,73.33,88,0.453,1610612752,NYK,New York Knicks,43,80,0.538,15,35,0.429,11,15,0.733,5,25,30,29,7.0,9,2,1,17,16,112,6.0,126.4,127.3,119.0,120.5,7.4,6.8,0.674,4.14,23.2,0.211,0.646,0.453,0.080,0.631,0.647,88.8,88.0,73.33,88,0.547,F,PF,28.0
87,NaN,2025-26,1629020,Jarred Vanderbilt,Jarred,1610612747,LAL,Los Angeles Lakers,22501170,2026-04-09T00:00:00,LAL @ GSW,W,25.596667,1,3,0.333,0,2,0.000,0,0,0.000,1,5,6,5,4,0,0,0,0,0,2,15,12.7,0,0,13.0,1,25:36,1,122.5,129.4,129.4,101.6,98.1,98.1,20.9,31.3,31.3,0.192,1.25,41.7,0.050,0.238,0.146,33.3,33.3,0.333,0.333,0.121,0.121,97.59,96.58,80.48,96.58,0.051,51,1.0,3.0,49,80,0.613,16,29,0.552,5,8,0.625,8,25,33,37,19.0,14,3,2,13,6,119,16.0,125.9,129.3,114.1,112.0,11.8,17.4,0.755,1.95,26.4,0.324,0.595,0.474,0.207,0.713,0.712,92.4,92.0,76.67,92,0.570,1610612744,GSW,Golden State Warriors,41,81,0.506,9,30,0.300,12,12,1.000,15,23,38,24,19.0,8,2,3,6,13,103,-16.0,114.1,112.0,125.9,129.3,-11.8,-17.4,0.585,1.26,18.0,0.405,0.676,0.526,0.207,0.562,0.597,92.4,92.0,76.67,92,0.430,NaN,PF,26.0
88,NaN,2025-26,1642880,Kam Jones,Kam,1610612754,IND,Indiana Pacers,22501167,2026-04-09T00:00:00,IND @ BKN,W,21.733333,2,7,0.286,0,2,0.000,0,0,0.000,1,2,3,6,4,0,0,1,2,1,4,9,12.6,0,0,13.0,1,21:44,1,116.8,119.6,119.6,99.7,97.9,97.9,17.2,21.7,21.7,0.286,1.50,35.3,0.045,0.083,0.065,23.5,23.5,0.286,0.286,0.208,0.203,102.96,102.70,85.58,102.70,0.016,46,2.0,7.0,51,98,0.520,8,31,0.258,13,18,0.722,13,53,66,43,12.0,4,5,5,16,21,123,29.0,117.2,118.3,87.6,90.4,29.7,27.9,0.843,3.58,26.9,0.260,0.828,0.579,0.115,0.561,0.581,106.1,104.0,86.67,104,0.714,1610612751,BKN,Brooklyn Nets,37,96,0.385,8,38,0.211,12,19,0.632,7,36,43,20,10.0,2,5,5,21,16,94,-29.0,87.6,90.4,117.2,118.3,-29.7,-27.9,0.541,2.00,14.8,0.172,0.740,0.421,0.096,0.427,0.450,106.1,104.0,86.67,104,0.286,NaN,SG,23.0
60,NaN,2025-26,1630551,Justin Champagnie,Justin,1610612764,WAS,Washing

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260410_102409.json


,home_team,away_team,commence_time,bookmakers
0,Atlanta Hawks,Cleveland Cavaliers,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Charlotte Hornets,Detroit Pistons,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Washington Wizards,Miami Heat,2026-04-10 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Boston Celtics,New Orleans Pelicans,2026-04-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Indiana Pacers,Philadelphia 76ers,2026-04-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-10 10:24:09
US latest pull: 2026-04-10 10:22:52


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Underdog,player_points,James Harden,Over,22.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
1,Underdog,player_points,James Harden,Under,22.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
2,Underdog,player_points,Evan Mobley,Over,18.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
3,Underdog,player_points,Evan Mobley,Under,18.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09
4,Underdog,player_points,Jonathan Kuminga,Over,12.5,-137,2026-04-10,2026-04-10T17:24:08Z,2026-04-10 10:24:09


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Micah Peavy: single positional indexer is out-of-bounds
[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Devin Carter: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,James Harden,AST,5.71,30.03,39.90,0.1537,0.1969,0.3426,0.88,5.91,13.67,"[0.2608922515001304, 0.1716443529007895, 0.217..."
1,Max Strus,AST,14.46,23.99,30.66,0.0581,0.1012,0.2744,0.84,2.43,8.41,"[0.2534854245880861, 0.0857632933104631, 0.093..."
2,Daniss Jenkins,AST,20.29,27.04,33.32,0.0866,0.1865,0.2943,1.76,5.04,9.81,"[0.2706359945872801, 0.1529051987767584, 0.328..."
3,Ausar Thompson,AST,17.84,26.61,35.30,0.0727,0.1251,0.2125,1.30,3.33,7.50,"[0.1921229586935638, 0.0, 0.124275062137531, 0..."
4,Miles Bridges,AST,14.23,29.46,37.44,0.0501,0.1105,0.2070,0.71,3.25,7.75,"[0.0899550224887556, 0.159846547314578, 0.0591..."
5,Jordan Poole,AST,19.00,30.30,37.21,0.1006,0.1616,0.2833,1.91,4.90,10.54,"[0.1140250855188141, 0.0602409638554216, 0.079..."
6,Tyrese Maxey,AST,7.97,30.11,41.28,0.1020,0.1792,0.2850,0.81,5.40,11.77,"[0.077359463641052, 0.1932100469224399, 0.2256..."
7,VJ Edgecombe,AST,27.53,36.50,40.36,0.0457,0.1314,0.2291,1.26,4.80,9.25,"[0.1448016217781639, 0.0, 0.2004008016032064, ..."
8,Adem Bona,AST,12.82,19.66,27.42,0.0639,0.0592,0.1251,0.82,1.16,3.43,"[0.1031991744066047, 0.0, 0.0633713561470215, ..."
9,Jalen Brunson,AST,6.09,28.74,38.61,0.1447,0.2292,0.3409,0.88,6.59,13.16,"[0.38860103626943, 0.4437869822485207, 0.16615..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
38,Cade Cunningham,AST,8.5,12.94,26.51,36.28,1.80,5.12,12.92,0.392,0.608
26,Darius Garland,AST,6.5,11.37,25.67,37.28,1.61,5.36,12.02,0.372,0.628
261,Desmond Bane,PTS,20.5,10.05,29.59,37.48,4.76,18.00,40.27,0.412,0.588
57,Derrick White,AST,5.5,12.20,28.54,37.48,0.79,4.16,9.81,0.249,0.751
27,Jrue Holiday,AST,6.5,16.26,29.15,38.46,1.71,5.13,11.59,0.376,0.625
149,Rui Hachimura,REB,3.5,16.38,27.14,34.23,0.42,4.13,9.91,0.399,0.601
65,Tre Johnson,REB,2.5,17.70,24.30,29.41,0.73,2.71,6.37,0.550,0.450
158,Evan Mobley,PTS,19.5,9.23,25.29,36.99,3.65,13.86,33.49,0.351,0.649
70,Tre Jones,REB,3.5,13.70,26.54,33.71,0.65,3.20,9.15,0.476,0.524
286,Royce O'Neale,PTS,9.5,15.17,22.02,28.99,2.47,7.71,22.02,0.319,0.681


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
232,Javon Small,PTS,16.5,15.46,22.03,27.41,3.81,10.51,26.73,0.146,0.854,PTS,Underdog,Utah Jazz,4.0,248.0,121.3,30.0,103.30,2.0,-137.0,-137.0,0.578,0.578,13.5,13.5,5.28,-3.0,-3.0,0.568,0.285,0.715,-50.70,23.69,0.0,0.3,0.27,0.15,24.88,4.46,0.18,0.05,8.00,2.0
169,Jalen Duren,PTS,18.5,13.87,28.50,37.46,4.54,13.53,32.77,0.465,0.535,PTS,Underdog,Charlotte Hornets,5.5,225.5,113.5,11.0,97.68,26.0,-114.0,-114.0,0.533,0.533,21.7,21.5,6.34,4.2,4.0,-0.662,0.746,0.254,40.04,-52.32,0.8,0.8,0.80,0.36,31.69,5.42,0.23,0.06,10.83,6.0
203,Tre Jones,PTS,17.5,13.70,26.54,33.71,4.07,11.21,29.26,0.448,0.552,PTS,Underdog,Orlando Magic,15.5,242.5,113.9,14.0,100.42,14.0,-137.0,-137.0,0.578,0.578,20.1,19.5,6.12,2.6,2.0,-0.425,0.665,0.335,15.04,-42.05,0.6,0.6,0.60,0.18,27.57,2.00,0.23,0.04,14.75,4.0
207,Victor Wembanyama,PTS,19.5,16.51,27.12,34.11,9.07,21.14,35.91,0.771,0.229,PTS,Underdog,Dallas Mavericks,-18.5,236.5,115.2,19.0,102.56,4.0,-116.0,-111.0,0.537,0.526,27.3,24.5,9.43,7.8,5.0,-0.827,0.796,0.204,48.22,-61.22,0.8,0.7,0.80,0.69,28.16,6.59,0.33,0.04,25.50,4.0
239,Jrue Holiday,PTS,17.5,16.26,29.15,38.46,4.88,13.80,32.67,0.405,0.596,PTS,Underdog,Los Angeles Clippers,-1.5,227.0,115.1,18.0,97.35,28.0,-106.0,-112.0,0.515,0.528,16.8,13.0,7.47,-0.7,-4.5,0.094,0.463,0.537,-10.02,1.65,0.6,0.4,0.33,0.27,31.05,5.28,0.22,0.06,18.67,3.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
222,Alperen Sengun,PTS,19.5,11.31,28.93,36.72,4.89,16.81,32.28,0.369,0.630,PTS,PrizePicks,Minnesota Timberwolves,-10.5,221.0,112.1,6.0,101.45,9.0,100.0,-106.0,0.500,0.515,21.4,21.5,9.66,1.9,2.0,-0.197,0.578,0.422,15.60,-17.99,0.4,0.5,0.40,0.46,32.96,4.82,0.24,0.05,25.83,6.0
176,Tre Johnson,PTS,13.5,17.70,24.30,29.41,6.19,14.82,29.16,0.407,0.593,PTS,PrizePicks,Miami Heat,16.5,245.5,113.7,12.0,104.24,1.0,-137.0,-137.0,0.578,0.578,10.4,11.0,3.75,-3.1,-2.5,0.827,0.204,0.796,-64.71,37.70,0.0,0.1,0.20,0.37,23.62,1.01,0.24,0.02,14.00,2.0
25,Maxime Raynaud,AST,1.5,19.71,29.16,35.64,0.02,1.24,3.70,0.430,0.570,AST,PrizePicks,Golden State Warriors,10.5,228.5,114.2,16.0,100.09,18.0,-137.0,-137.0,0.578,0.578,1.6,2.0,1.35,0.1,0.5,-0.074,0.529,0.471,-8.49,-18.52,0.6,0.6,0.67,0.40,31.38,4.76,0.20,0.06,1.00,3.0
94,Ausar Thompson,REB,4.5,17.84,26.61,35.30,2.19,7.10,14.59,0.686,0.314,REB,PrizePicks,Charlotte Hornets,5.5,225.5,113.5,11.0,97.68,26.0,107.0,-139.0,0.483,0.582,5.4,5.0,1.84,0.9,0.5,-0.489,0.688,0.312,42.42,-46.35,0.6,0.6,0.53,0.60,28.90,4.73,0.14,0.05,6.25,4.0
179,Neemias Queta,PTS,10.5,17.55,24.98,30.68,4.69,10.39,24.67,0.655,0.345,PTS,PrizePicks,New Orleans Pelicans,-16.5,224.0,117.3,23.0,101.25,11.0,-137.0,-137.0,0.578,0.578,12.6,12.5,5.17,2.1,2.0,-0.406,0.658,0.342,13.83,-40.84,0.8,0.7,0.53,0.26,27.63,4.62,0.14,0.04,6.00,1.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
141,Ryan Rollins,REB,4.5,22.37,33.38,36.66,0.74,4.70,9.46,0.449,0.551,REB,Betr DFS,Brooklyn Nets,-9.5,218.0,117.8,24.0,97.59,27.0,103.0,-133.0,0.493,0.571,4.0,3.5,2.67,-0.5,-1.0,0.187,0.426,0.574,-13.52,0.56,0.2,0.2,0.27,0.31,31.13,4.49,0.28,0.08,2.50,4.0
204,Collin Sexton,PTS,21.5,19.96,25.55,33.01,9.60,18.61,39.32,0.370,0.630,PTS,Betr DFS,Orlando Magic,15.5,242.5,113.9,14.0,100.42,14.0,-137.0,-137.0,0.578,0.578,20.6,19.5,4.22,-0.9,-2.0,0.213,0.416,0.584,-28.04,1.03,0.2,0.4,0.40,0.22,28.29,4.03,0.25,0.03,18.60,5.0
193,Brandon Ingram,PTS,21.5,8.63,27.15,37.34,4.07,17.33,35.77,0.260,0.740,PTS,Betr DFS,New York Knicks,6.5,219.0,112.4,8.0,97.82,25.0,-105.0,-118.0,0.512,0.541,18.9,18.0,8.25,-2.6,-3.5,0.315,0.376,0.624,-26.59,15.28,0.6,0.3,0.40,0.49,31.42,3.73,0.24,0.04,25.75,4.0
84,Deandre Ayton,REB,7.5,9.08,25.72,36.13,1.86,9.27,16.42,0.499,0.501,REB,Betr DFS,Phoenix Suns,2.5,218.0,113.0,10.0,98.25,24.0,-109.0,-115.0,0.522,0.535,6.2,6.0,3.43,-1.3,-1.5,0.379,0.352,0.648,-32.51,21.15,0.2,0.3,0.47,0.63,25.00,4.18,0.16,0.05,10.86,7.0
247,Collin Gillespie,PTS,13.5,18.33,27.71,35.36,5.16,11.43,26.01,0.286,0.714,PTS,Betr DFS,Los Angeles Lakers,-2.5,218.0,116.1,20.0,99.25,22.0,-137.0,-137.0,0.578,0.578,9.5,11.0,6.02,-4.0,-2.5,0.664,0.253,0.747,-56.23,29.23,0.0,0.2,0.20,0.34,26.77,5.35,0.17,0.04,15.60,5.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
10,Scottie Barnes,AST,6.5,11.66,27.70,37.48,1.42,5.46,12.55,0.527,0.473,AST,DraftKings Pick6,New York Knicks,6.5,219.0,112.4,8.0,97.82,25.0,110.0,-129.0,0.476,0.563,9.2,10.0,3.94,2.7,3.5,-0.685,0.753,0.247,58.13,-56.15,0.4,0.7,0.60,0.34,29.62,4.20,0.21,0.04,4.38,8.0
171,Daniss Jenkins,PTS,10.5,20.29,27.04,33.32,4.80,13.32,27.36,0.845,0.155,PTS,DraftKings Pick6,Charlotte Hornets,5.5,225.5,113.5,11.0,97.68,26.0,103.0,-133.0,0.493,0.571,18.6,18.5,6.10,8.1,8.0,-1.328,0.908,0.092,84.32,-83.88,0.8,0.9,0.73,0.34,33.96,6.61,0.23,0.03,2.00,2.0
9,Jalen Brunson,AST,7.5,6.09,28.74,38.61,0.88,6.59,13.16,0.393,0.607,AST,DraftKings Pick6,Toronto Raptors,-6.5,219.0,112.1,7.0,99.29,21.0,-101.0,-127.0,0.502,0.559,8.7,8.5,2.98,1.2,1.0,-0.403,0.657,0.343,30.75,-38.69,0.8,0.7,0.73,0.42,36.30,4.11,0.29,0.05,6.88,8.0
195,Josh Hart,PTS,11.5,10.84,27.76,36.20,2.09,9.86,21.29,0.319,0.681,PTS,DraftKings Pick6,Toronto Raptors,-6.5,219.0,112.1,7.0,99.29,21.0,-114.0,-111.0,0.533,0.526,14.3,14.0,9.45,2.8,2.5,-0.296,0.616,0.384,15.64,-27.01,0.4,0.6,0.53,0.63,31.13,5.02,0.17,0.07,16.50,8.0
61,Nickeil Alexander-Walker,REB,3.5,15.84,29.42,39.26,0.40,3.27,9.28,0.442,0.558,REB,DraftKings Pick6,Cleveland Cavaliers,-8.0,233.5,114.0,15.0,100.68,13.0,100.0,-125.0,0.500,0.556,3.5,4.0,1.08,0.0,0.5,0.000,0.500,0.500,0.00,-10.00,0.8,0.7,0.60,0.38,34.89,4.96,0.22,0.03,3.80,5.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
198,Ryan Rollins,PTS,20.5,22.37,33.38,36.66,8.14,18.44,39.85,0.512,0.488,PTS,Betr DFS,Brooklyn Nets,-9.5,218.0,117.8,24.0,97.59,27.0,-137.0,-137.0,0.578,0.578,21.2,21.5,7.13,0.7,1.0,-0.098,0.539,0.461,-6.76,-20.25,0.8,0.5,0.40,0.20,31.13,4.49,0.28,0.08,8.00,4.0
31,LeBron James,AST,9.5,6.42,29.75,39.85,0.85,7.28,14.12,0.245,0.755,AST,DraftKings Pick6,Phoenix Suns,2.5,218.0,113.0,10.0,98.25,24.0,100.0,-115.0,0.500,0.535,8.7,9.5,3.86,-0.8,0.0,0.207,0.418,0.582,-16.40,8.81,0.6,0.5,0.33,0.29,34.31,4.12,0.22,0.05,6.29,7.0
192,Karl-Anthony Towns,PTS,18.5,13.81,27.81,36.51,7.35,17.80,42.35,0.566,0.434,PTS,Underdog,Toronto Raptors,-6.5,219.0,112.1,7.0,99.29,21.0,-103.0,-110.0,0.507,0.524,20.2,21.0,4.37,1.7,2.5,-0.389,0.651,0.349,28.30,-33.37,0.6,0.7,0.67,0.67,29.25,4.13,0.27,0.06,21.75,8.0
170,Duncan Robinson,PTS,9.5,15.46,24.49,34.92,3.86,10.13,23.70,0.759,0.241,PTS,Betr DFS,Charlotte Hornets,5.5,225.5,113.5,11.0,97.68,26.0,-116.0,-111.0,0.537,0.526,13.7,13.0,4.00,4.2,3.5,-1.050,0.853,0.147,58.83,-72.06,1.0,0.9,0.80,0.58,27.16,2.65,0.16,0.04,12.67,6.0
6,Tyrese Maxey,AST,6.5,7.97,30.11,41.28,0.81,5.40,11.77,0.347,0.653,AST,PrizePicks,Indiana Pacers,-15.5,232.5,117.9,25.0,101.71,8.0,-114.0,-105.0,0.533,0.512,6.1,7.0,2.85,-0.4,0.5,0.140,0.444,0.556,-16.65,8.55,0.6,0.5,0.60,0.48,36.27,5.49,0.27,0.04,5.00,5.0


### Get top EVs for 2 legs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 156  |  Pairs: 564  |  Slate: 9  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 96  |  Pairs: 515  |  Slate: 10  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 109  |  Pairs: 347  |  Slate: 8  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 119  |  Pairs: 341  |  Slate: 9  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [18]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 156  |  Triples: 16696  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 96  |  Triples: 12603  |  Slate: 8  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 119  |  Triples: 8621  |  Slate: 7  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 109  |  Triples: 8780  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
